In [1]:
from kbel.disambiguators import Disambiguator
from kbel.disambiguators.naive import NaiveDisambiguator
from kbel.disambiguators.similarity import SimilarityDisambiguator
from kbel.core.mention import Mention
from kbel.core.mention import EntityType
from kbel.knowledge_sources import KnowledgeSource
# import logging
# logging.basicConfig(level=logging.DEBUG)

/Users/marcelomachado/Documents/projects/kbel/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Using `naive` disambiguator to link entities from Wikidata

In [2]:
kbel = Disambiguator(strategy_name='naive')
results = kbel.disambiguate(
    mention=Mention(label='rock', text='', entity_type=EntityType.ITEM),
    ks=KnowledgeSource('wikidata', limit=10))
display (*results)

('rock music',
 'popular music genre',
 Item(IRI('http://www.wikidata.org/entity/Q11399')))

### Using `similarity` disambiguator to link entities from Wikidata

In [4]:
kbel = Disambiguator(strategy_name='sim')
results = kbel.disambiguate(
    ks=KnowledgeSource('wikidata-wapi', limit=10),
    mention=Mention(label='Rock', text='Rock is a stone', entity_type=EntityType.ITEM),
    limit=2)
display (*results)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6166.60it/s]


('stone',
 'rock; building material',
 Item(IRI('http://www.wikidata.org/entity/Q22731')))

('Rock',
 'male given name',
 Item(IRI('http://www.wikidata.org/entity/Q60589667')))

In [12]:
results = kbel.disambiguate(
    ks=KnowledgeSource('wikidata-wapi', limit=10),
    mention=Mention(label='instance of', text='Rock is a stone', entity_type=EntityType.PROPERTY)
)

display(*results)

('instance of',
 'type to which this subject corresponds/belongs. Different from P279 (subclass of); for example: K2 is an instance of mountain; volcano is a subclass of mountain',
 Property(IRI('http://www.wikidata.org/entity/P31'), None))

('subproperty of',
 'all resources related by this property are also related by that property',
 Property(IRI('http://www.wikidata.org/entity/P1647'), None))

('individual of taxon',
 'the taxon of an individual named organism (animal, plant)',
 Property(IRI('http://www.wikidata.org/entity/P10241'), None))

In [6]:
kbel = Disambiguator(strategy_name='sim')
results = kbel.disambiguate(
    ks=KnowledgeSource('dbpedia', limit=10),
    mention=Mention(label='Rock', text='Rock is a stone to build buildings', entity_type=EntityType.ITEM),
    limit=2)
display (*results)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6287.85it/s]


('Soft rock', '', Item(IRI('http://dbpedia.org/resource/Soft_rock')))

('Rock music', '', Item(IRI('http://dbpedia.org/resource/Rock_music')))

### Using `LLM` disambiguator to link entities from Wikidata

Instantiating LLM Disambiguator with IBM WatsonX's models

In [7]:
import os
import dotenv
dotenv.load_dotenv()

kbel = Disambiguator(
    'llm',
    model_name='meta-llama/llama-3-3-70b-instruct',
    model_provider='ibm',
    model_apikey=os.environ['LLM_API_KEY'],
    model_endpoint=os.environ['LLM_API_ENDPOINT'],
)

ModuleNotFoundError: No module named 'dotenv'

In [11]:
results = kbel.disambiguate(
    mention=Mention(label='Rock', text='A rock can be used in construction to mimic the appearance and durability of natural stone.', entity_type=EntityType.ITEM),
    ks=KnowledgeSource('wikidata-wapi', limit=100))

display (*results)

('stone',
 'rock; building material',
 Item(IRI('http://www.wikidata.org/entity/Q22731')))

('rock mechanics',
 'theoretical and applied science of the mechanical behavior of rock and rock masses; compared to geology',
 Item(IRI('http://www.wikidata.org/entity/Q1191291')))

('rock art',
 'human-made markings on natural stone',
 Item(IRI('http://www.wikidata.org/entity/Q1211146')))

('petroglyph',
 'images carved on a rock surface as a form of rock art',
 Item(IRI('http://www.wikidata.org/entity/Q42195')))

('rock',
 'mass of stone projecting out of the ground or water',
 Item(IRI('http://www.wikidata.org/entity/Q1404150')))

('rock formation',
 'stone mass',
 Item(IRI('http://www.wikidata.org/entity/Q631305')))

('instrumental rock',
 'type of rock music',
 Item(IRI('http://www.wikidata.org/entity/Q1650296')))

('rock',
 'naturally occurring solid aggregate of one or more minerals or mineraloids',
 Item(IRI('http://www.wikidata.org/entity/Q8063')))

('Rock',
 'male given name',
 Item(IRI('http://www.wikidata.org/entity/Q60589667')))

('rock asphalt',
 'petroleum product',
 Item(IRI('http://www.wikidata.org/entity/Q202251')))

LLM disambiguator uses [LangChain ChatModels](https://python.langchain.com/docs/integrations/providers/). Below, we use IBM WatsonX's models

In [ ]:
from langchain_ibm import ChatWatsonx
model = ChatWatsonx(
    model_id='meta-llama/llama-3-3-70b-instruct',
    apikey=os.environ['LLM_API_KEY'], # type: ignore
    url=os.environ['LLM_API_ENDPOINT'], # type: ignore
    project_id=os.environ['WATSONX_PROJECT_ID'],
    temperature=0.0
)
kbel = Disambiguator(strategy_name='llm', model=model)

In [ ]:
results = kbel.disambiguate(
    mention=Mention(label='Rock', text='A rock can be used in construction to mimic the appearance and durability of natural stone.'),
    ks=KnowledgeSource('wikidata-wapi', limit=100))
display (*results)

('rock',
 'mass of stone projecting out of the ground or water',
 Item(IRI('http://www.wikidata.org/entity/Q1404150')))

('rock',
 'naturally occurring solid aggregate of one or more minerals or mineraloids',
 Item(IRI('http://www.wikidata.org/entity/Q8063')))